In [1]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    r"C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs"
)

ADMISSIONS_PATH = r"C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs\mimic-iv-3.1\mimic-iv-3.1\hosp\admissions.csv.gz"
DIAGNOSES_PATH = r"C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs\mimic-iv-3.1\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"

#DIAGNOSES_PATH = r"C:\path\to\mimiciv\hosp\diagnoses_icd.csv.gz"
# ADMISSIONS_PATH = r"C:\path\to\mimiciv\hosp\admissions.csv.gz"

# Load new longitudinal cohort
cohort = pd.read_parquet(
    OUTPUT_DIR / "cohort_labels_longitudinal.parquet"
)

# Load diagnoses
diagnoses = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=[
        "subject_id",
        "hadm_id",
        "seq_num",
        "icd_code",
        "icd_version"
    ]
)

# Load admissions
admissions = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=[
        "subject_id",
        "hadm_id",
        "admittime"
    ]
)

admissions["admittime"] = pd.to_datetime(
    admissions["admittime"]
)

# Attach each patient's index admission time
patient_admissions = admissions.merge(
    cohort[
        [
            "subject_id",
            "index_hadm_id",
            "index_admittime"
        ]
    ],
    on="subject_id",
    how="inner"
)

patient_admissions["index_admittime"] = pd.to_datetime(
    patient_admissions["index_admittime"]
)

# Keep ONLY admissions occurring BEFORE the index admission
patient_admissions = patient_admissions[
    patient_admissions["admittime"]
    < patient_admissions["index_admittime"]
].copy()

# Sort visits chronologically
patient_admissions = patient_admissions.sort_values(
    [
        "subject_id",
        "admittime",
        "hadm_id"
    ]
)

# Number prior visits
patient_admissions["visit_number"] = (
    patient_admissions
    .groupby("subject_id")
    .cumcount()
    + 1
)

# Add diagnosis codes
dx_sequences = diagnoses.merge(
    patient_admissions[
        [
            "subject_id",
            "hadm_id",
            "visit_number",
            "admittime"
        ]
    ],
    on=[
        "subject_id",
        "hadm_id"
    ],
    how="inner"
)

dx_sequences = dx_sequences.rename(
    columns={
        "seq_num": "code_rank_within_visit"
    }
)

# Sort diagnosis sequence
dx_sequences = dx_sequences.sort_values(
    [
        "subject_id",
        "visit_number",
        "code_rank_within_visit"
    ]
).reset_index(drop=True)

# Save
output_file = (
    OUTPUT_DIR /
    "dx_sequences_longitudinal.parquet"
)

dx_sequences.to_parquet(
    output_file,
    index=False
)

print("Rows:", len(dx_sequences))

print(
    "Patients:",
    dx_sequences["subject_id"].nunique()
)

print("\nVisit distribution:")

print(
    dx_sequences
    .groupby("subject_id")["visit_number"]
    .max()
    .describe()
)

print("\nSaved:")
print(output_file)

Rows: 1189896
Patients: 24346

Visit distribution:
count    24346.000000
mean         3.725622
std          5.454783
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max        237.000000
Name: visit_number, dtype: float64

Saved:
C:\Users\oluwa\OneDrive\Desktop\CoC Files\Current Literature\DNTransformerLLMsMIMIC\outputs-20260805T024105Z-1-001\outputs\dx_sequences_longitudinal.parquet
